In [6]:
import pandas as pd
import numpy as np
import pickle

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

In [2]:
file_path = "notebooks/MarketPrices2010-2025.xlsx"

df = pd.read_excel(file_path)

# Remove completely empty columns
df = df.dropna(axis=1, how="all")

# Fill missing values if any
df = df.ffill()

print(df.head())

   Year  Coking Coal Price (INR/t)  \
0  2010                       5941   
1  2011                      13076   
2  2012                       9078   
3  2013                       9376   
4  2014                       7320   

   Electricity Price - HT Industrial (INR/kWh)  Steam Price (INR/GJ)  \
0                                         5.50                   180   
1                                         5.65                   192   
2                                         5.80                   204   
3                                         5.95                   216   
4                                         6.50                   228   

   Carbon Price - Coal Cess / GST Compensation Cess (INR/tonne coal)  \
0                                                 50                   
1                                                 50                   
2                                                 50                   
3                                                 

In [3]:
future_years = np.arange(
    df["Year"].min(),
    2051
).reshape(-1,1)

predicted_df = pd.DataFrame({
    "Year": future_years.flatten()
})

In [4]:
for column in df.columns[1:]:

    X = df[["Year"]]

    y = df[column]

    # Linear Regression
    linear = LinearRegression()
    linear.fit(X,y)

    pred_linear = linear.predict(X)

    rmse_linear = np.sqrt(
        mean_squared_error(y,pred_linear)
    )
    # Polynomial Regression
    poly = Pipeline([
        ("poly", PolynomialFeatures(degree=2)),
        ("linear", LinearRegression())
    ])

    poly.fit(X,y)

    pred_poly = poly.predict(X)

    rmse_poly = np.sqrt(
        mean_squared_error(y,pred_poly)
    )

    # Select best model
    if rmse_linear <= rmse_poly:
        best_model = linear
    else:
        best_model = poly

    predicted_df[column] = best_model.predict(future_years)


c:\MAC\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but PolynomialFeatures was fitted with feature names
  warnings.warn(
c:\MAC\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but PolynomialFeatures was fitted with feature names
  warnings.warn(
c:\MAC\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\MAC\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\MAC\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\MAC\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does no

In [5]:
predicted_df.to_excel(
    "notebooks/predicted_market_prices.xlsx",
    index=False
)

print("\nPrediction completed.")
print(predicted_df.tail())


Prediction completed.
    Year  Coking Coal Price (INR/t)  \
36  2046               41171.533100   
37  2047               42162.607903   
38  2048               43154.166984   
39  2049               44146.210343   
40  2050               45138.737979   

    Electricity Price - HT Industrial (INR/kWh)  Steam Price (INR/GJ)  \
36                                     7.867748                 612.0   
37                                     7.928911                 624.0   
38                                     7.990103                 636.0   
39                                     8.051325                 648.0   
40                                     8.112577                 660.0   

    Carbon Price - Coal Cess / GST Compensation Cess (INR/tonne coal)  \
36                                        1115.294118                   
37                                        1144.558824                   
38                                        1173.823529                   
39         

In [7]:
# Save the best model
with open('best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)